# vectortileserver — demo & bridge test

Renders a PMTiles vector layer in `ipyleaflet`, exercising both fixes this package ships:

- **#2 — conversion:** a vector source → PMTiles via `tippecanoe` (when it's on `PATH`), keeping
  every point at every zoom level.
- **#3 — browser reachability:** the `jupyter_loopback` bridge that lets tiles load in sandboxed
  webviews (Voila / SEPAL / VS Code / Colab), where `http://localhost:<port>` is unreachable.

**Setup:** `micromamba env create -f environment.yml` (installs `tippecanoe` + the package), or
`pip install vectortileserver` with `tippecanoe` on your `PATH`. `tippecanoe` is *optional* for this
notebook — without it we synthesize a tiny PMTiles archive so the bridge/render demo still runs.

In [1]:
import shutil

HAS_TIPPECANOE = shutil.which("tippecanoe") is not None
print(
    "tippecanoe:",
    "found -> convert a GeoJSON (exercises #2)"
    if HAS_TIPPECANOE
    else "not found -> synthesize a .pmtiles (exercises #3 only)",
)

tippecanoe: found -> convert a GeoJSON (exercises #2)


## 1. Get a PMTiles archive

With `tippecanoe` we generate ~2,000 random points carrying a `map_code` category and let
`TileClient` convert them (retaining every point). Without it we write a minimal one-tile archive
directly, so the rest of the notebook still runs. Both helpers are inline so the notebook is
self-contained.

In [2]:
import gzip
import json
import random
import tempfile
from pathlib import Path

CATEGORIES = [0, 1, 2, 3]
work = Path(tempfile.mkdtemp(prefix="vectortileserver-demo-"))


def write_points(path, count=2000, spread=0.5, seed=0):
    """A GeoJSON FeatureCollection of random points, each with a map_code."""
    rng = random.Random(seed)
    features = [
        {
            "type": "Feature",
            "properties": {"map_code": rng.choice(CATEGORIES)},
            "geometry": {
                "type": "Point",
                "coordinates": [rng.uniform(0, spread), rng.uniform(0, spread)],
            },
        }
        for _ in range(count)
    ]
    path.write_text(json.dumps({"type": "FeatureCollection", "features": features}))
    return path


def write_minimal_pmtiles(path):
    """A valid single-tile PMTiles archive, without shelling out to tippecanoe."""
    import mapbox_vector_tile
    from pmtiles.tile import Compression, TileType
    from pmtiles.writer import Writer

    corners = [(1024, 1024, 0), (3072, 3072, 1), (1024, 3072, 2), (3072, 1024, 3)]
    tile = mapbox_vector_tile.encode(
        [
            {
                "name": "points",
                "features": [
                    {"geometry": f"POINT({x} {y})", "properties": {"map_code": code}}
                    for x, y, code in corners
                ],
            }
        ]
    )
    with open(path, "wb") as f:
        writer = Writer(f)
        writer.write_tile(0, gzip.compress(tile))
        writer.finalize(
            {
                "tile_type": TileType.MVT,
                "tile_compression": Compression.GZIP,
                "internal_compression": Compression.GZIP,
                "min_zoom": 0,
                "max_zoom": 0,
                "min_lon_e7": 0,
                "min_lat_e7": 0,
                "max_lon_e7": int(0.5 * 1e7),
                "max_lat_e7": int(0.5 * 1e7),
            },
            {"vector_layers": [{"id": "points", "fields": {"map_code": "Number"}}]},
        )
    return path

In [3]:
from vectortileserver import TileClient

if HAS_TIPPECANOE:
    source = write_points(work / "points.geojson", count=2000)
else:
    source = write_minimal_pmtiles(work / "points.pmtiles")

client = TileClient(source, allowed_directories=[work])
print("data source:", source.name)
print("pmtiles    :", client.pmtiles_path)

2026-07-24 11:37:58,982 - VECTORTILES - DEBUG - Initializing tile client with data source: /tmp/vectortileserver-demo-y3uuxdxe/points.geojson
2026-07-24 11:37:58,983 - VECTORTILES - DEBUG - Processing vector data: /tmp/vectortileserver-demo-y3uuxdxe/points.geojson -> /tmp/vectortileserver-demo-y3uuxdxe
2026-07-24 11:37:59,417 - VECTORTILES - DEBUG - Converted data to PMTiles: /tmp/vectortileserver-demo-y3uuxdxe/points.pmtiles


INFO:     Started server process [4021658]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://localhost:8001 (Press CTRL+C to quit)


2026-07-24 11:38:00,410 - VECTORTILES - DEBUG - PMTiles server running at http://localhost:8001
data source: points.geojson
pmtiles    : /tmp/vectortileserver-demo-y3uuxdxe/points.pmtiles


## 2. Inspect the client

`pmtiles_url` is what the browser fetches. Inside a Jupyter kernel it is a root-relative **proxy
path** (`/vectortileserver-proxy/<port>/…`); outside one it is the raw loopback URL.
`create_leaflet_layer()` auto-installs the `jupyter_loopback` comm bridge for `server_port`, so tiles
still reach the browser in webviews whose origin is not the jupyter-server.

In [4]:
print("server_port  :", client.server_port)
print("client_prefix:", client.client_prefix)  # None outside a Jupyter kernel
print("pmtiles_url  :", client.pmtiles_url)
print("layers       :", client.list_layers())
print("bounds       :", client.bounds)
print("center       :", client.center)

server_port  : 8001
client_prefix: None
pmtiles_url  : http://localhost:8001/pmtiles?filePath=/tmp/vectortileserver-demo-y3uuxdxe/points.pmtiles
layers       : ['points']
bounds       : {'left': 0.000127, 'bottom': 0.00012, 'right': 0.499941, 'top': 0.499916}
center       : (0.250018, 0.25003400000000003)



## 3. Render as a layer

`vts.open_async(source)` converts (if needed), serves, and returns a ready `ipyleaflet` layer that
knows its own bounds — no client, no hand-rolled style, no hardcoded zoom. `m.fit_bounds(layer.bounds)`
frames the map on it. Scroll to zoom and use the layers control (top-right) to toggle the layer; every
point stays put at every zoom level.

In [ ]:
import vectortileserver as vts
from ipyleaflet import LayersControl, Map

# One call: convert (off-thread) + serve + return a ready ipyleaflet layer that
# knows its own extent. Points render from the geometry-agnostic default style.
layer = await vts.open_async(source)
layer.name = "points (PMTiles)"

m = Map(scroll_wheel_zoom=True)
m.add(layer)
m.add(LayersControl(position="topright"))
if layer.bounds:   # None for a single-point/degenerate archive; skip fitting
    m.fit_bounds(layer.bounds)   # zoom to the layer's own bounds — no hardcoded zoom
m

In [ ]:
if layer.bounds:   # None for a single-point/degenerate archive; skip fitting
    m.fit_bounds(layer.bounds)   # zoom to the layer's own bounds — no hardcoded zoom

## Multiple datasets = just more layers

`open_async` returns a normal ipyleaflet layer, so N datasets is N layers on one
map. `m.fit_bounds(vts.default_workspace().bounds())` frames all of them at once
(union of their bounds). One shared tile server serves them all.

In [ ]:
# A second dataset on the same map (reuses the same server + workspace).
layer2 = await vts.open_async(source, style=vts.single_symbol_style(color="#1E90FF"))
layer2.name = "points (blue)"
m.add(layer2)
bounds = vts.default_workspace().bounds()   # union of everything opened
if bounds:
    m.fit_bounds(bounds)                    # frames both

# Restyle on the fly: the frontend won't repaint on `.style =`, so swap the
# cheap layer (same cached archive, no reconversion).
recolored = layer.with_style(vts.single_symbol_style(color="#e41a1c"))
recolored.name = layer.name
m.remove(layer)
m.add(recolored)
layer = recolored  # rebind so re-running this cell removes the current layer, not a stale one

## 4. Smoke check (headless)

Even without a browser, confirm the server honors HTTP Range requests — the mechanism PMTiles relies
on. This hits the raw loopback server directly (not the proxy path), so it works from the kernel.

In [ ]:
from urllib.parse import quote

import httpx

url = f"{client.server_url}/pmtiles?filePath={quote(str(client.pmtiles_path), safe='/')}"
resp = httpx.get(url, headers={"range": "bytes=0-127"})

print("status       :", resp.status_code, "(expect 206)")
print("content-range:", resp.headers.get("content-range"))
print("bytes        :", len(resp.content))
assert resp.status_code == 206 and len(resp.content) == 128
print("\nRange serving OK.")

## 5. Verify the bridge in a browser

In JupyterLab the map above already loads tiles (same origin). To prove the bridge for **sandboxed
webviews**, serve this notebook with Voila:

```bash
voila examples/demo.ipynb
```

Open it, pan/zoom the map, and watch the browser **Network** tab: tiles should load with **no direct
`127.0.0.1` / `localhost` requests** — every tile travels over the jupyter-server proxy or the
`jupyter_loopback` comm bridge.

To see the contrast, disable the bridge and reload — tiles should then fail to load in a sandboxed
webview:

```bash
VECTORTILESERVER_DISABLE_JUPYTER_LOOPBACK=1 voila examples/demo.ipynb
```